## Context Optimisation for MCP Servers with Databricks Agent

### Installing Utilities and Libraries

In [ ]:
%pip install \
    databricks-sdk==0.49.0 \
    openai-agents==0.22.0 \
    mcp==2.0.0 \
    databricks-mcp==0.9.2 \
    "mlflow>=3.1"

### Restarting the Python Environment

In [ ]:
dbutils.library.restartPython()

### Set up your Environment

In [ ]:
from databricks.sdk import WorkspaceClient

# Get Databricks runtime authentication
w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = w.config.host.rstrip("/")

### Define your MCP Server

In [ ]:
from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient

mcp_server_url = "YOUR-MCP-SERVER-URL-GOES-HERE"

### Execute the Agent with MCP Server

In [ ]:
from agents import (
    Agent,
    Runner,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    ToolSearchTool,
    set_tracing_disabled
)
from agents.mcp import MCPServerStreamableHttp


# Create an OpenAI-compatible client for Databricks
client = AsyncOpenAI(
    api_key=token,
    base_url=f"{workspace_host}/serving-endpoints"
)

# Configure the Databricks model
model = OpenAIChatCompletionsModel(
    model="databricks-claude-sonnet-4-5",
    openai_client=client
)


async with MCPServerStreamableHttp(
    name="MSLearn-MCP-Server",

    params={
        "url": mcp_server_url,
        "headers": {
            "Authorization": f"Bearer {token}"
        }
    },

    # Enable staged tool discovery
    defer_loading=True

) as mcp_server:

    # Don't send Agents SDK traces to OpenAI
    set_tracing_disabled(True)

    # Create agent
    agent = Agent(
        name="MSLearn-Agent",

        instructions=(
            "You are a helpful AI assistant with access to "
            "the Microsoft Learn MCP Server. Discover and use "
            "MCP tools when appropriate."
        ),

        model=model,

        mcp_servers=[
            mcp_server
        ],

        # Allows the agent to discover deferred tools
        tools=[
            ToolSearchTool()
        ]
    )

    # Run agent
    result = await Runner.run(
        agent,
        "Give me a Microsoft Learn learning path for Azure Databricks."
    )

    print(result.final_output)